# RUN_POLICY - 运行预训练策略 (生成评估视频)

评估脚本 `lerobot_eval.py` 会在模拟环境中运行策略，并生成 MP4 视频作为结果，**不会**启动 Rerun 实时预览。

> **提示**：以下所有命令已自动使用 `conda run -n lerobot` 前缀，无需手动激活环境即可直接运行。
>
> **无头服务器渲染配置**：如果在无头服务器（SSH 远程）运行 LIBERO/robosuite 环境，请先设置环境变量：
> ```bash
> export MUJOCO_GL=egl
> export PYOPENGL_PLATFORM=egl
> export EGL_DEVICE_ID=0
> ```

---

## 核心参数说明

- `--policy.path`: 预训练模型的 Hugging Face ID 或本地路径
- `--env.type`: 环境类型 (`pusht`, `aloha`, `libero`)
- `--env.task`: 环境任务名称 (如 `libero_10`, `libero_spatial`)
- `--eval.batch_size`: 并行评估的环境数量
- `--eval.n_episodes`: 总共评估的回合数
- `--policy.device`: 运行策略的设备 (`cuda` 或 `cpu`)
- `--seed`: 设置随机种子 (默认 1000)
- `--eval.use_async_envs`: (可选) 是否使用异步环境运行，默认为 `false`



> **环境说明**：本 Notebook 中所有 `python` 命令均已前置 `conda run -n lerobot`，确保在正确的环境中执行。
> 如果你是在 `lerobot` 环境中启动的 Jupyter，也可以手动去掉 `conda run -n lerobot` 前缀直接运行。


In [ ]:
# 安装 libero 依赖（运行 LIBERO 环境必需）
# CMAKE_POLICY_VERSION_MINIMUM=3.5 绕过 cmake 4.x 与 egl_probe 的兼容性问题
!CMAKE_POLICY_VERSION_MINIMUM=3.5 conda run -n lerobot pip install --no-build-isolation -e "./lerobot[libero]"


---

## 1. Diffusion Policy on PushT (2D 模拟)

最简单的入门示例，运行 Diffusion Policy 完成推方块任务。


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/diffusion_pusht \
--env.type=pusht \
--eval.batch_size=10 \
--eval.n_episodes=10 \
--policy.use_amp=false \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


## 2. ACT on Aloha (双臂模拟)

运行 Action Chunking Transformer (ACT) 完成双臂传递方块任务。


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/act_aloha_sim_transfer_cube_human \
--env.type=aloha \
--env.task=AlohaTransferCube-v0 \
--eval.batch_size=1 \
--eval.n_episodes=10 \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


## 3. Pi0 (VLA) on Libero (3D 模拟)

使用 Pi0 (1.4B) 视觉语言动作模型运行 LIBERO 基准测试。


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0_libero_finetuned \
--env.type=libero \
--env.task=libero_10 \
--eval.batch_size=1 \
--eval.n_episodes=10 \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


### LIBERO 多品类评估

LIBERO 包含 **5 个任务品类**，共 **130 个任务**：

| 品类 | 标识符 | 任务数 | 最大步数 |
|:---|:---|:---|:---|
| LIBERO-Spatial | `libero_spatial` | 10 | 280 |
| LIBERO-Object | `libero_object` | 10 | 280 |
| LIBERO-Goal | `libero_goal` | 10 | 300 |
| LIBERO-90 | `libero_90` | 90 | 400 |
| LIBERO-Long | `libero_10` | 10 | 520 |

**评估单个品类**：


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0_libero_finetuned \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=10

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


**同时评估多个品类（逗号分隔）**：


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0_libero_finetuned \
--env.type=libero \
--env.task=libero_spatial,libero_object,libero_goal,libero_10 \
--eval.batch_size=1 \
--eval.n_episodes=10

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


---

## 4. 更多 VLA 模型一键运行

### 4.1 Pi0 (Physical Intelligence 0) - 通用机器人基础模型

> **依赖安装**：见下方代码 cell（可直接点击运行）


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[pi]"


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0_libero_finetuned \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


### 4.2 Pi0.5 (开放世界通用模型)

> **依赖安装**：见下方代码 cell（可直接点击运行）


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[pi]"


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi05_libero_finetuned \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


### 4.3 X-VLA (软提示跨形态适应)

> **依赖安装**：见下方代码 cell（可直接点击运行）


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[xvla]"


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/xvla-libero \
--env.type=libero \
--env.task=libero_spatial,libero_goal,libero_10 \
--eval.batch_size=1 \
--eval.n_episodes=1 \
--env.episode_length=800

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


### 4.4 NVIDIA GR00T N1.5 - 人形机器人专用

> **依赖安装**：见下方代码 cell（可直接点击运行）
>
> **注意**：目前 Hugging Face 上**没有**公开的预训练 GR00T Libero 权重，以下代码仅作为运行指令模板。


In [ ]:
!conda run -n lerobot pip install flash-attn==2.7.4.post1 --no-build-isolation
!conda run -n lerobot pip install -e "./lerobot[groot]"


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=aractingi/bimanual-handover-groot-10k \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


### 4.5 SmolVLA (450M 参数) - 轻量级高效

> **依赖安装**：见下方代码 cell（可直接点击运行）


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[smolvla]"


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=HuggingFaceVLA/smolvla_libero \
--env.type=libero \
--env.task=libero_10 \
--eval.batch_size=2 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


---

## 5. VLA 模型对比总结

| 模型 | 参数量 | 显存需求 | 特点 | 适用场景 |
|:---|:---|:---|:---|:---|
| **SmolVLA** | 450M | 4-8GB | 轻量、快速 | 消费级显卡、边缘计算 |
| **Pi0** (π₀) | 1.4B | 24GB+ | 基础通用模型 | 通用机器人控制 |
| **Pi0.5** | ~1.4B | 24GB+ | 开放世界泛化 | 跨环境、未见场景 |
| **X-VLA** | 0.9B+ | 16GB+ | 跨形态适应 | 多种机器人形态适配 (软提示) |
| **GR00T** | 3B | 40GB+ | 人形机器人 | NVIDIA 生态、人形操作 |

---

## 6. 查看任意评估结果

运行完成后，结果保存在 `outputs/eval/{日期}/{时间}_{任务名}` 目录下：

- **视频文件**：`outputs/eval/.../videos/` 包含每个评估回合的 MP4 录像
- **指标数据**：`eval_info.json` 包含成功率和奖励统计

可随时用以下命令查看任意一次评估结果（浏览器自动打开）：

```bash
# 查看最新结果
python scripts/view_eval.py

# 查看指定目录
python scripts/view_eval.py outputs/eval/2026-04-23/15-12-29_libero_pi0
```
